In [ ]:
# Apply Custom CSS Styling (Section 1)
from IPython.display import display, HTML

display(HTML("""
<!-- MathJax config -->
<script type="text/x-mathjax-config">
MathJax.Hub.Config({
  tex2jax: {
    inlineMath: [['$','$'], ['\\\\(','\\\\)']],
    displayMath: [['$$','$$'], ['\\\\[','\\\\]']],
    processEscapes: true,
    processEnvironments: true,
    skipTags: ['script', 'noscript', 'style', 'textarea', 'pre']
  },
  TeX: {
    equationNumbers: { autoNumber: "AMS" },
    extensions: ["AMSmath.js", "AMSsymbols.js"]
  }
});
</script>
<script src="https://cdn.jsdelivr.net/npm/mathjax@2/MathJax.js?config=TeX-AMS_HTML"></script>

<style>
    body {
        font-family: 'Helvetica Neue', Arial, sans-serif;
        line-height: 1.5;
        max-width: 100%;
        margin: 0 auto;
        padding: 20px;
    }
    h1 {
        color: #2c3e50;
        text-align: center;
        padding-bottom: 15px;
        border-bottom: 2px solid #3498db;
        margin-bottom: 30px;
    }
    h2 {
        color: #3498db;
        margin-top: 30px;
        padding-bottom: 10px;
        border-bottom: 1px solid #eee;
    }
    h3 {
        color: #2980b9;
        margin-top: 25px;
    }
    .section {
        margin: 30px 0;
        padding: 20px;
        background: #f8f9fa;
        border-radius: 5px;
        border-left: 5px solid #3498db;
    }
    .author-info {
        display: block;
        width: 100%;
        background: #f8f9fa;
        padding: 20px;
        border-radius: 8px;
        margin: 30px auto;
        border-left: 5px solid #3498db;
        box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        clear: both;
        box-sizing: border-box;
    }
    .funding-info {
        display: block;
        width: 100%;
        background: #f0f7fa;
        padding: 20px;
        border-radius: 8px;
        margin: 30px auto;
        border-left: 5px solid #2980b9;
        font-size: 0.95em;
        box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        clear: both;
        box-sizing: border-box;
    }
    .theory {
        background: #f8f9fa;
        padding: 15px;
        border-radius: 5px;
        margin: 20px 0;
        border-left: 5px solid #3498db;
    }
    .widget-area {
        background: #f1f8ff;
        padding: 20px;
        border-radius: 5px;
        margin: 30px 0;
        border: 1px solid #d1e5f9;
    }
    .footer {
        margin-top: 50px;
        padding-top: 20px;
        border-top: 1px solid #eee;
        text-align: left;
        font-size: 0.9em;
        color: #7f8c8d;
    }
    .jupyter-widgets-output-area {
        width: 100% !important;
        max-width: 100% !important;
        overflow-x: auto !important;
    }
    .jupyter-matplotlib-figure {
        width: 100% !important;
        max-width: 100% !important;
    }
    .jupyter-matplotlib-canvas-container {
        width: 100% !important;
    }
    canvas.jupyter-matplotlib-canvas {
        max-width: 100% !important;
        height: auto !important;
    }
    .widget-output {
        width: 100% !important;
        overflow-x: hidden !important;
    }
    .figure {
        max-width: 100% !important;
        margin: 0 auto !important;
    }
</style>
"""))

In [ ]:
# --- Title and Author/Funding Info ---
display(HTML("""
<h1>HTSegregation Pyiron Workflow Demonstrator</h1>
<div class="author-info">
    <h2>Authors</h2>
    <p><strong>Adapted for NFDI Demonstrator</strong></p>
    <p><strong>Original Workflow:</strong> pyiron_workflow_atomistics team</p>
    <p><strong>Styling:</strong> Based on NFDI Thermal Homogenization Demonstrator</p>
</div>
<div class="funding-info">
    <h2>Funding Acknowledgment</h2>
    <p>This notebook demonstrates grain boundary segregation calculations using pyiron_workflow_atomistics and LAMMPS, styled for NFDI-MatWerk demonstrators.</p>
</div>
"""))

In [ ]:
# --- Grain Boundary Segregation Workflow ---
display(HTML("""
<div class="section">
    <h1>Grain Boundary Segregation Workflow</h1>
    <p>This notebook demonstrates a complete workflow for studying grain boundary (GB) segregation in materials using <code>pyiron_workflow_atomistics</code> and the LAMMPS calculation engine.</p>
    <h2>Overview</h2>
    <ul>
        <li><strong>Lattice Optimization:</strong> Optimize the bulk lattice parameter</li>
        <li><strong>Solution Energy:</strong> Calculate the solution energy of a solute in bulk</li>
        <li><strong>GB Search:</strong> Find suitable grain boundary structures</li>
        <li><strong>Pure GB Study:</strong> Analyze the pure grain boundary properties</li>
        <li><strong>Segregation Study:</strong> Calculate segregation energies at GB sites</li>
    </ul>
</div>
"""))

In [ ]:
import os
from typing import Union, Optional, Tuple

import numpy as np
import pandas as pd

# ASE imports
from ase.build import bulk, stack
from ase.lattice.cubic import BodyCenteredCubic as bcc

# Pyiron workflow imports
import pyiron_workflow as pwf
from pyiron_workflow import Workflow
from pyiron_workflow_atomistics.dataclass_storage import CalcInputMinimize
from pyiron_workflow_atomistics.bulk import optimise_cubic_lattice_parameter
from pyiron_workflow_lammps.engine import LammpsEngine
from pyiron_workflow_atomistics.structure_manipulator.tools import create_supercell, create_supercell_with_min_dimensions
from pyiron_workflow_atomistics.structure_manipulator.tools import substitutional_swap_one_site

from pyiron_workflow_atomistics.gb.gb_code.searcher import get_gb_code_df_with_structures
from pyiron_workflow_atomistics.calculator import calculate_structure_node
# Pymatgen imports
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor

%load_ext autoreload
%autoreload 2

In [ ]:
# --- Verify Installation ---
# display(HTML("""
# <div class="section">
#     <h2>Verify Installation</h2>
#     <p>Let's verify that the <code>pyiron_workflow_atomistics</code> package is correctly installed and accessible.</p>
# </div>
# """))

import pyiron_workflow_atomistics
# print(pyiron_workflow_atomistics.__file__)

In [ ]:
# --- Step 1: Workflow Definition and Lattice Optimization ---
display(HTML("""
<div class="section">
    <h2>Step 1: Workflow Definition and Lattice Optimization</h2>
    <p>In this step, we:</p>
    <ul>
        <li>Initialize the workflow with a name and clean workspace</li>
        <li>Create the initial structure using ASE's bulk builder for Fe (BCC)</li>
        <li>Setup the LAMMPS calculation engine with:
            <ul>
                <li>EAM/FS potential for Al-Fe system</li>
                <li>Minimization settings (no cell relaxation)</li>
                <li>Working directory structure</li>
            </ul>
        </li>
        <li>Optimize the lattice parameter by:
            <ul>
                <li>Applying small strains to the structure</li>
                <li>Calculating energies at each strain</li>
                <li>Fitting an equation of state (Birch-Murnaghan)</li>
                <li>Extracting the equilibrium lattice parameter</li>
            </ul>
        </li>
    </ul>
    <p>This gives us the relaxed bulk structure with the optimal lattice parameter for subsequent calculations.</p>
</div>
"""))

wf = Workflow("gb_segregation", delete_existing_savefiles=True)
structure = bulk("Fe", a=2.85, cubic=True)

# Engine definition - minimization for structure relaxation
inp = CalcInputMinimize()
inp.relax_cell = False  # Don't relax the cell
Engine = LammpsEngine(EngineInput=inp)
Engine.working_directory = "gb_segregation"
Engine.lammps_log_filepath = "minimize.log"
Engine.command = "lmp -in in.lmp -log minimize.log"
Engine.input_script_pair_style = "eam/fs"
potential_path = os.getcwd() + "/Al-Fe.eam.fs"
Engine.path_to_model = potential_path

os.makedirs("calculations", exist_ok=True)
os.chdir("calculations")
# Optimize the cubic lattice parameter
wf.opt_cubic_cell = optimise_cubic_lattice_parameter(
    structure=structure,
    name="Fe",
    crystalstructure="bcc",
    calculation_engine=Engine,
    parent_working_directory="opt_cubic_cell",
    rattle=0.1,
    strain_range=(-0.02, 0.02),
    num_points=10,
    eos_type="birchmurnaghan",
)

In [ ]:
# --- Step 2: Solution Energy Calculation ---
display(HTML("""
<div class="section">
    <h2>Step 2: Solution Energy Calculation</h2>
    <p>The solution energy tells us how favorable it is to dissolve a solute atom (Al) into the bulk host material (Fe).</p>
    <p>In this step, we:</p>
    <ul>
        <li>Create a large supercell (12×12×12 Å minimum dimensions) to avoid spurious interactions between periodic images of the solute</li>
        <li>Calculate the energy of pure bulk supercell</li>
        <li>Create a supercell with one Al solute by substituting one Fe atom at position 0</li>
        <li>Calculate the energy with the solute present</li>
        <li>Compute the solution energy as: <br>
            <span style="font-family:monospace;">E_solution = E_with_solute - E_pure_bulk</span>
        </li>
    </ul>
    <p>A positive solution energy indicates that dissolving Al in Fe is energetically unfavorable in the bulk.</p>
    <p>This value will be used later to calculate segregation energies at grain boundaries.</p>
</div>
"""))

from pyiron_workflow_atomistics.utils import duplicate_engine

# Create duplicate engines for bulk calculations
Engine_bulk = duplicate_engine.node_function(Engine, "solution_energy_bulk")
Engine_bulk.working_directory = "solution_energy_bulk"

# Create supercell to avoid solute-solute interactions
wf.supercell = create_supercell_with_min_dimensions(
    base_structure=wf.opt_cubic_cell.outputs.equil_struct,
    min_dimensions=[12, 12, 12],
)
wf.supercell_calc = calculate_structure_node(wf.supercell, calculation_engine=Engine_bulk)

# Create supercell with solute (Fe -> Al substitution)
Engine_bulk_solute = duplicate_engine.node_function(Engine, "solution_energy_solute")
Engine_bulk_solute.working_directory = "solution_energy_solute"

wf.supercell_with_1sol = substitutional_swap_one_site(
    base_structure=wf.supercell,
    defect_site=0,
    new_symbol="Al",
)
wf.supercell_with_1sol_calc = calculate_structure_node(
    structure=wf.supercell_with_1sol,
    calculation_engine=Engine_bulk_solute,
)

# Calculate solution energy
@pwf.as_function_node("solution_energy")
def calculate_soln_energy(bulk_structure_energy, soln_structure_energy):
    return soln_structure_energy - bulk_structure_energy

wf.soln_energy = calculate_soln_energy(
    bulk_structure_energy=wf.supercell_calc.outputs.calc_output.final_energy,
    soln_structure_energy=wf.supercell_with_1sol_calc.outputs.calc_output.final_energy,
)

In [ ]:
# --- Step 3: Grain Boundary Search ---
display(HTML("""
<div class="section">
    <h2>Step 3: Grain Boundary Search</h2>
    <p>Now we search for suitable grain boundary structures using the GBCode algorithm.</p>
    <p>In this step, we:</p>
    <ul>
        <li>Search for CSL (Coincidence Site Lattice) grain boundaries along three different crystallographic axes: [111], [110], and [100]</li>
        <li>Filter structures based on:
            <ul>
                <li>Sigma values (up to Σ10)</li>
                <li>Maximum number of atoms (≤100)</li>
                <li>Minimum grain dimensions (10 Å in-plane, 15 Å along grain)</li>
            </ul>
        </li>
        <li>Generate atomistic structures for each valid GB configuration</li>
        <li>Remove duplicates to get a unique set of GB structures</li>
    </ul>
    <p>The output is a DataFrame containing various GB structures with different misorientations, planes, and atomic arrangements.</p>
    <p>For this workflow, we'll select the first GB structure from the results for detailed analysis.</p>
</div>
"""))

# Search GBCode for valid CSL GB structures
wf.gb_code_df = get_gb_code_df_with_structures(
    axes_list=[np.array([1, 1, 1]), np.array([1, 1, 0]), np.array([1, 0, 0])],
    sigma_limit=10,
    lim_plane_index=3,
    max_atoms=100,
    max_workers=None,
    deduplicate=True,
    element="Fe",
    basis="bcc",
    lattice_param=wf.opt_cubic_cell.outputs.a0,
    equil_volume_per_atom=wf.opt_cubic_cell.outputs.equil_volume_per_atom,
    min_inplane_gb_length=10,
    req_length_grain=15,
    grain_length_axis=0,
)

In [ ]:
# --- Step 4: Pure Grain Boundary Study ---
display(HTML("""
<div class="section">
    <h2>Step 4: Pure Grain Boundary Study</h2>
    <p>This comprehensive step analyzes the properties of the pure (undecorated) grain boundary.</p>
    <ul>
        <li><strong>GB Plane Identification:</strong> Uses Voronoi site featurization to identify atomic environments, compares GB region atoms to bulk templates, locates the GB plane position automatically.</li>
        <li><strong>Grain Length Optimization:</strong> Coarse and fine scan to find the optimal grain extension that minimizes GB energy.</li>
        <li><strong>Cleavage Energy Calculation:</strong> Identifies viable cleavage planes near the GB, calculates rigid and relaxed cleavage energy.</li>
        <li><strong>Structure Preparation:</strong> Adds vacuum layer (20 Å) for surface calculations, expands cell to minimum dimensions (6×6 Å in-plane), prepares the final GB structure for segregation studies.</li>
    </ul>
    <p><strong>Output:</strong> Optimized pure GB structure with known GB plane location, GB energy, and cleavage properties.</p>
</div>
"""))

from pyiron_workflow_atomistics.dataclass_storage import CalcInputStatic
from pyiron_workflow_atomistics.gb.gb_study import pure_gb_study
from pyiron_workflow_atomistics.gb.dataclass_storage import CleaveGBStructureInput, PlotCleaveInput
from pyiron_workflow_atomistics.featurisers import voronoiSiteFeaturiser

# Create engines for GB study
Engine_gb = duplicate_engine.node_function(Engine, "gb_study")
Engine_gb.working_directory = "gb_study"

# Static engine for cleavage energy calculations
inp_static = CalcInputStatic()
Engine_static = LammpsEngine(EngineInput=inp_static)
Engine_static.working_directory = "pure_grain_boundary_study"
Engine_static.lammps_log_filepath = "static.log"
Engine_static.command = "lmp -in in.lmp -log static.log"
Engine_static.input_script_pair_style = "eam/fs"
Engine_static.path_to_model = potential_path

# Run pure GB study
wf.pure_gb_study = pure_gb_study(
    gb_structure=wf.gb_code_df.outputs.gb_code_df_with_structures.iloc[0].structure,
    equil_bulk_volume=wf.opt_cubic_cell.outputs.equil_volume_per_atom,
    equil_bulk_energy=wf.opt_cubic_cell.outputs.equil_energy_per_atom,
    extensions_stage1=np.linspace(-0.2, 0.8, 3),
    extensions_stage2=np.linspace(-0.05, 0.05, 5),
    calculation_engine=Engine_gb,
    static_calculation_engine=Engine_static,
    length_interpolate_min_n_points=5,
    gb_normal_axis="c",
    vacuum_length=20,
    min_inplane_cell_lengths=[6, 6, None],
    featuriser=voronoiSiteFeaturiser,
    approx_frac=0.5,
    tolerance=5.0,
    bulk_offset=10.0,
    slab_thickness=2.0,
    featuriser_kwargs=None,
    n_bulk=10,
    threshold_frac=0.3,
    CleaveGBStructure_Input=CleaveGBStructureInput(tol=0.3),
    PlotCleave_Input=PlotCleaveInput()
)

In [ ]:
# --- Step 5: Segregation Study ---
display(HTML("""
<div class="section">
    <h2>Step 5: Segregation Study</h2>
    <p>This is the main segregation calculation where we determine which GB sites are energetically favorable for Al segregation.</p>
    <ul>
        <li><strong>Site Identification and Deduplication:</strong> Use SOAP descriptors and PCA to group sites with similar environments and select representatives.</li>
        <li><strong>Segregation Energy Calculations:</strong>
            <ul>
                <li>Create a structure with Al substituted at each unique site</li>
                <li>Calculate the total energy using LAMMPS</li>
                <li>Compute segregation energy as:<br>
                    <span style="font-family:monospace;">E_seg = (E_GB+Al - E_GB_pure) - E_solution</span>
                </li>
            </ul>
        </li>
        <li><strong>Interpretation:</strong>
            <ul>
                <li>Negative E_seg: Segregation is favorable (Al prefers GB over bulk)</li>
                <li>Positive E_seg: Segregation is unfavorable (Al prefers bulk)</li>
            </ul>
        </li>
    </ul>
    <p>The magnitude indicates the strength of the segregation tendency.</p>
</div>
"""))

from pyiron_workflow_atomistics.gb.segregation import calculate_substitutional_segregation_GB, get_unique_sites_SOAP

# Identify unique sites using SOAP descriptors
wf.site_duplicate_df = get_unique_sites_SOAP(
    structure=wf.pure_gb_study.outputs.pure_grain_boundary_structure_vacuum,
    defect_sites=wf.pure_gb_study.outputs.gb_plane_analysis_dict["extended_sel_indices"],
    r_cut=6.0,
    n_max=10,
    l_max=10,
    n_jobs=-1,
    periodic=True,
    pca_variance_threshold=0.999,
    similarity_threshold=0.99999
)

# Create engine for segregation calculations
Engine_segregation = duplicate_engine.node_function(Engine, "segregation_study")
Engine_segregation.working_directory = "segregation_study"

# Calculate substitutional segregation energies
wf.gb_seg_calcs = calculate_substitutional_segregation_GB(
    structure=wf.pure_gb_study.outputs.pure_grain_boundary_structure_vacuum,
    defect_sites=wf.site_duplicate_df.outputs.unique_sites_list,
    element="Al",
    structure_basename="pureGB_Fe_seg",
    parent_dir="gb_seg_lammps",
    calculation_engine=Engine_segregation,
    unique_sites_df=wf.site_duplicate_df.outputs.df,
    df_filename="seg_calcs_df.pkl",
)

In [ ]:
# --- Workflow Execution ---
display(HTML("""
<div class="section">
    <h2>Workflow Execution</h2>
    <ul>
        <li>Executes all nodes that can be computed with available inputs</li>
    </ul>
    <p><strong>Expected Output:</strong></p>
    <ul>
        <li>Progress bars for GB structure generation</li>
        <li>LAMMPS calculation logs</li>
        <li>Status messages for each workflow stage</li>
    </ul>
    <p><strong>Execution Time:</strong> This may take several minutes to hours depending on:</p>
    <ul>
        <li>Number of GB structures found</li>
        <li>Number of unique segregation sites</li>
        <li>Computational resources available</li>
        <li>Type of engine used (eam potential/ml potential/)</li>
    </ul>
</div>
"""))

# wf.run()

import logging
import io
from contextlib import redirect_stdout, redirect_stderr

# quiet global and specific noisy loggers
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("pyiron_workflow").setLevel(logging.WARNING)
logging.getLogger("pyiron_workflow_atomistics").setLevel(logging.WARNING)

# run and capture textual output in memory; keep display outputs (figures) intact
buf_out = io.StringIO()
buf_err = io.StringIO()
try:
    with redirect_stdout(buf_out), redirect_stderr(buf_err):
        wf.run()
except Exception:
    # show captured text for debugging
    print("Workflow failed; stdout:")
    print(buf_out.getvalue())
    print("Workflow failed; stderr:")
    print(buf_err.getvalue())
    raise

In [ ]:
df = wf.gb_seg_calcs.outputs.gb_seg_calcs_df.value.copy()
df["energy"] = df.calc_output.apply(lambda x: x.final_energy)
df["Eseg"] = (
    df.energy
    - wf.pure_gb_study.outputs.pure_grain_boundary_structure_vacuum_energy.value
    - wf.soln_energy.outputs.solution_energy.value
)

import numpy as np

gb_pos = wf.pure_gb_study.outputs.gb_plane_analysis_dict.value["gb_cart"]

def get_site_gb_distance(row):
    struct = row["structure"]
    rep_idx = row["rep"]
    dist = np.round(np.abs(struct.positions[rep_idx][2] - gb_pos), 1)
    return dist

df["dist_GB"] = df.apply(get_site_gb_distance, axis=1)

# Show the first 10 rows in a styled section
display(HTML("""
<div class="section">
    <h2>Segregation Results Table</h2>
    <p>Below are the first 10 segregation results, sorted by distance from the grain boundary:</p>
</div>
"""))
display(HTML(df.sort_values(by="dist_GB").head(10).to_html(classes='table table-striped', border=0)))

In [ ]:
os.chdir("..")
import shutil
shutil.rmtree("calculations", ignore_errors=True)